In [9]:
pip install pandas matplotlib geopandas contextily matplotlib-scalebar cartopy

# Improved Claude Code

In [12]:
import matplotlib
matplotlib.use('Agg')  # Safe backend for Colab/headless
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.lines import Line2D


# ============================================================
# FIGURE 1: US Data Center Density with ERCOT Highlight
# ============================================================

def create_us_map():
    # -- Data --
    markets = [
        # name, lat, lon, capacity_MW, tier
        ("Northern\nVirginia",   39.04, -77.49, 4500, "saturated"),
        ("Silicon\nValley",      37.35, -121.96, 1800, "saturated"),
        ("Chicago",              41.88, -87.63,  1200, "saturated"),
        ("Atlanta",              33.75, -84.39,   800, "growth"),
        ("Phoenix",              33.45, -112.07, 1000, "growth"),
        ("Portland",             45.54, -122.90,  600, "growth"),
        ("Dallas-\nFort Worth",  32.78, -96.80,  1500, "ercot_hyper"),
        ("Austin",               30.27, -97.74,   500, "ercot_hyper"),
        ("San Antonio",          29.42, -98.49,   400, "ercot_hyper"),
        ("Houston",              29.76, -95.37,   350, "ercot_emerging"),
    ]

    tier_props = {
        "saturated":       {"color": "#8e99a4", "label": "Stable / Saturated"},
        "growth":          {"color": "#4a90d9", "label": "High Growth"},
        "ercot_hyper":     {"color": "#d63031", "label": "Hyper-Growth (ERCOT)"},
        "ercot_emerging":  {"color": "#e17055", "label": "Emerging (ERCOT)"},
    }

    # Albers Equal-Area — the standard US map projection
    proj = ccrs.AlbersEqualArea(central_longitude=-96, central_latitude=37.5,
                                 standard_parallels=(29.5, 45.5))

    fig, ax = plt.subplots(figsize=(12, 7.5), subplot_kw={"projection": proj})

    # -- Geographic features --
    ax.set_extent([-125, -66, 24, 50], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.OCEAN, facecolor='#f0f4f8', zorder=0)
    ax.add_feature(cfeature.LAND, facecolor='#fafafa', edgecolor='none', zorder=0)
    ax.add_feature(cfeature.STATES, edgecolor='#d0d0d0', linewidth=0.4, zorder=1)
    ax.add_feature(cfeature.BORDERS, edgecolor='#a0a0a0', linewidth=0.6, zorder=1)
    ax.add_feature(cfeature.COASTLINE, edgecolor='#a0a0a0', linewidth=0.5, zorder=1)
    ax.add_feature(cfeature.LAKES, facecolor='#f0f4f8', edgecolor='#c0c0c0', linewidth=0.3, zorder=1)

    # Define text_effect here, before its first use
    text_effect = [pe.withStroke(linewidth=2.5, foreground='white')]

    # -- ERCOT highlight (clean bounding box around Texas metros) --
    ercot_lons = [-104.8, -93.5, -93.5, -104.8, -104.8]
    ercot_lats = [25.8,   25.8,  36.8,   36.8,   25.8]
    ax.plot(ercot_lons, ercot_lats, transform=ccrs.PlateCarree(),
            color='#d63031', linewidth=2, linestyle='--', alpha=0.7, zorder=2)
    ax.text(-99.0, 37.5, "ERCOT Service Territory",
            transform=ccrs.PlateCarree(),
            fontsize=8.5, fontstyle='italic', fontweight='bold',
            color='#c0392b', ha='center', zorder=10,
            path_effects=text_effect)

    # -- Plot bubbles --
    scale = 0.08  # controls bubble area per MW
    for name, lat, lon, mw, tier in markets:
        props = tier_props[tier]
        ax.scatter(lon, lat, transform=ccrs.PlateCarree(),
                   s=mw * scale, c=props['color'], alpha=0.8,
                   edgecolors='white', linewidths=0.8, zorder=5)

    # -- Labels --
    label_nudge = {
        "Northern\nVirginia":  (2.0,  1.5),
        "Silicon\nValley":     (-3.5, -2.5),
        "Chicago":             (2.0,  1.5),
        "Atlanta":             (2.5, -1.5),
        "Phoenix":             (-4.0, -2.0),
        "Portland":            (2.0,  1.2),
        "Dallas-\nFort Worth": (3.0,  1.0),
        "Austin":              (-4.5,  0.0),
        "San Antonio":         (-5.5, -1.0),
        "Houston":             (2.5, -1.5),
    }
    for name, lat, lon, mw, tier in markets:
        dx, dy = label_nudge.get(name, (2, 1))
        is_ercot = 'ercot' in tier
        ax.annotate(
            name, xy=(lon, lat), xytext=(lon + dx, lat + dy),
            transform=ccrs.PlateCarree(),
            fontsize=7 if not is_ercot else 7.5,
            fontweight='bold' if is_ercot else 'normal',
            color='#2c3e50',
            arrowprops=dict(arrowstyle='-', color='#999999', lw=0.6,
                            connectionstyle='arc3,rad=0.1'),
            path_effects=text_effect, zorder=10,
            annotation_clip=True
        )

    # -- Legend --
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor=p['color'],
               markersize=10, markeredgecolor='white', markeredgewidth=0.5,
               label=p['label'])
        for p in tier_props.values()
    ]
    # Size reference
    for ref_mw in [500, 1500, 4000]:
        legend_elements.append(
            Line2D([0], [0], marker='o', color='w', markerfacecolor='#cccccc',
                   markersize=np.sqrt(ref_mw * scale) * 0.85,
                   markeredgecolor='#999999', markeredgewidth=0.4,
                   label=f'{ref_mw} MW')
        )

    leg = ax.legend(handles=legend_elements, loc='lower right',
                    fontsize=8, title='Market Trajectory', title_fontsize=9,
                    frameon=True, fancybox=True, framealpha=0.95,
                    edgecolor='#cccccc', borderpad=1.0,
                    handletextpad=0.8, labelspacing=0.9)
    leg.get_frame().set_linewidth(0.5)

    plt.tight_layout(pad=0.5)
    plt.savefig("us_texas_datacenters.pdf", bbox_inches='tight', facecolor='white')
    plt.savefig("us_texas_datacenters.png", dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print("Generated: us_texas_datacenters.pdf  &  .png")


# ============================================================
# FIGURE 2: Global Hyperscale Data Center Corridors
# ============================================================

def create_global_map():
    hubs = [
        # name, lat, lon, capacity_MW, cluster
        ("N. Virginia",     39.04,  -77.49,  5000, "namerica"),
        ("Texas (ERCOT)",   31.50,  -97.00,  2500, "namerica"),
        ("W. Coast",        38.50,  -121.50, 2500, "namerica"),
        ("Chicago",         41.88,  -87.63,  1500, "namerica"),
        ("Frankfurt",       50.11,   8.68,   1400, "europe"),
        ("London",          51.51,  -0.13,   1600, "europe"),
        ("Amsterdam",       52.37,   4.90,   1200, "europe"),
        ("Paris",           48.86,   2.35,    900, "europe"),
        ("Dublin",          53.35,  -6.26,   1100, "europe"),
        ("Tokyo",           35.68, 139.65,   1500, "apac"),
        ("Singapore",        1.35, 103.82,   1100, "apac"),
        ("Sydney",         -33.87, 151.21,    900, "apac"),
        ("Beijing",         39.90, 116.41,   2200, "apac"),
        ("São Paulo",      -23.55, -46.63,    500, "emerging"),
        ("Mumbai",          19.08,  72.88,    600, "emerging"),
        ("Johannesburg",   -26.20,  28.04,    300, "emerging"),
    ]

    cluster_props = {
        "namerica": {"color": "#27ae60", "label": "North America"},
        "europe":   {"color": "#2980b9", "label": "Europe (FLAP-D)"},
        "apac":     {"color": "#c0392b", "label": "Asia-Pacific"},
        "emerging": {"color": "#f39c12", "label": "Emerging Markets"},
    }

    proj = ccrs.Robinson(central_longitude=10)

    fig, ax = plt.subplots(figsize=(14, 7), subplot_kw={"projection": proj})

    # -- Geographic features --
    ax.set_global()
    ax.add_feature(cfeature.OCEAN, facecolor='#f0f4f8', zorder=0)
    ax.add_feature(cfeature.LAND, facecolor='#f5f5f5', edgecolor='none', zorder=0)
    ax.add_feature(cfeature.BORDERS, edgecolor='#d5d5d5', linewidth=0.3, zorder=1)
    ax.add_feature(cfeature.COASTLINE, edgecolor='#b0b0b0', linewidth=0.4, zorder=1)

    # -- Plot bubbles --
    scale = 0.07
    for name, lat, lon, mw, cluster in hubs:
        props = cluster_props[cluster]
        ax.scatter(lon, lat, transform=ccrs.PlateCarree(),
                   s=mw * scale, c=props['color'], alpha=0.8,
                   edgecolors='white', linewidths=0.7, zorder=5)

    # -- Labels (selected major hubs to avoid clutter) --
    text_effect = [pe.withStroke(linewidth=2, foreground='white')]
    labels_to_show = {
        "N. Virginia":   (5,  3),
        "Texas (ERCOT)": (-8, -4),
        "W. Coast":      (-10, -4),
        "Frankfurt":     (3,  3),
        "London":        (-6, 3),
        "Tokyo":         (4,  3),
        "Singapore":     (5, -3),
        "Beijing":       (5,  3),
        "Sydney":        (5, -3),
        "São Paulo":     (5, -4),
    }
    for name, lat, lon, mw, cluster in hubs:
        if name in labels_to_show:
            dx, dy = labels_to_show[name]
            ax.annotate(
                name, xy=(lon, lat),
                xytext=(lon + dx, lat + dy),
                transform=ccrs.PlateCarree(),
                fontsize=6.5, color='#2c3e50',
                arrowprops=dict(arrowstyle='-', color='#aaaaaa', lw=0.5),
                path_effects=text_effect, zorder=10,
                annotation_clip=True
            )

    # -- Legend --
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor=p['color'],
               markersize=9, markeredgecolor='white', markeredgewidth=0.5,
               label=p['label'])
        for p in cluster_props.values()
    ]
    leg = ax.legend(handles=legend_elements, loc='lower left',
                    fontsize=8, title='Data Center Cluster', title_fontsize=9,
                    frameon=True, fancybox=True, framealpha=0.95,
                    edgecolor='#cccccc', borderpad=0.8)
    leg.get_frame().set_linewidth(0.5)

    plt.tight_layout(pad=0.5)
    plt.savefig("global_datacenters.pdf", bbox_inches='tight', facecolor='white')
    plt.savefig("global_datacenters.png", dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print("Generated: global_datacenters.pdf  &  .png")


# ============================================================
if __name__ == "__main__":
    create_us_map()
    create_global_map()
    print("\nDone. Use the .pdf files in LaTeX for vector quality.")

Generated: us_texas_datacenters.pdf  &  .png
Generated: global_datacenters.pdf  &  .png

Done. Use the .pdf files in LaTeX for vector quality.


# Old Code

In [ ]:
"""
Improved Figures 1 and 2 for "Megawatts to Zettaflops" paper.
Run in Google Colab or any environment with internet access.

Requirements: pip install pandas matplotlib geopandas contextily matplotlib-scalebar
"""

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import contextily as ctx
import numpy as np

# ============================================================
# FIGURE 1: US Data Center Density with ERCOT Highlight
# ============================================================

def create_us_map():
    us_data = [
        # Tier 1 / Saturated
        {"Market": "Northern Virginia",  "Lat": 39.0438, "Lon": -77.4874, "Capacity_MW": 4500, "Tier": "Stable/Saturated"},
        {"Market": "Silicon Valley",     "Lat": 37.3541, "Lon": -121.9552,"Capacity_MW": 1800, "Tier": "Stable/Saturated"},
        {"Market": "Chicago",            "Lat": 41.8781, "Lon": -87.6298, "Capacity_MW": 1200, "Tier": "Stable/Saturated"},
        # High Growth
        {"Market": "Atlanta",            "Lat": 33.7490, "Lon": -84.3880, "Capacity_MW": 800,  "Tier": "High Growth"},
        {"Market": "Phoenix",            "Lat": 33.4484, "Lon": -112.074, "Capacity_MW": 1000, "Tier": "High Growth"},
        {"Market": "Portland/Hillsboro", "Lat": 45.5428, "Lon": -122.900, "Capacity_MW": 600,  "Tier": "High Growth"},
        # ERCOT Hyper-Growth
        {"Market": "Dallas-Fort Worth",  "Lat": 32.7767, "Lon": -96.7970, "Capacity_MW": 1500, "Tier": "Hyper-Growth (ERCOT)"},
        {"Market": "Austin",             "Lat": 30.2672, "Lon": -97.7431, "Capacity_MW": 500,  "Tier": "Hyper-Growth (ERCOT)"},
        {"Market": "San Antonio",        "Lat": 29.4241, "Lon": -98.4936, "Capacity_MW": 400,  "Tier": "Hyper-Growth (ERCOT)"},
        # ERCOT Emerging
        {"Market": "Houston",            "Lat": 29.7604, "Lon": -95.3698, "Capacity_MW": 350,  "Tier": "Emerging (ERCOT)"},
    ]

    df = pd.DataFrame(us_data)
    gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.Lon, df.Lat), crs="EPSG:4326").to_crs(epsg=3857)

    # Color and style mapping
    tier_style = {
        "Stable/Saturated":     {"color": "#7f8c8d", "zorder": 3},
        "High Growth":          {"color": "#3498db", "zorder": 4},
        "Hyper-Growth (ERCOT)": {"color": "#e74c3c", "zorder": 5},
        "Emerging (ERCOT)":     {"color": "#e67e22", "zorder": 5},
    }

    fig, ax = plt.subplots(figsize=(14, 9))

    # -- Draw ERCOT boundary highlight (approximate bounding box) --
    # ERCOT roughly covers central/east Texas
    ercot_lons = [-104.5, -93.5, -93.5, -104.5, -104.5]
    ercot_lats = [25.8, 25.8, 36.5, 36.5, 25.8]
    ercot_gdf = gpd.GeoDataFrame(
        geometry=[gpd.points_from_xy(ercot_lons, ercot_lats).unary_union.convex_hull],
        crs="EPSG:4326"
    ).to_crs(epsg=3857)
    ercot_gdf.boundary.plot(ax=ax, edgecolor='#e74c3c', linewidth=2.5, linestyle='--', alpha=0.6, zorder=2)

    # -- Plot bubbles by tier --
    for tier, style in tier_style.items():
        subset = gdf[gdf['Tier'] == tier]
        if len(subset) == 0:
            continue
        ax.scatter(
            subset.geometry.x, subset.geometry.y,
            s=subset['Capacity_MW'] * 0.5,
            c=style['color'], alpha=0.75,
            edgecolors='white', linewidth=1.2,
            zorder=style['zorder'],
            label=tier
        )

    # -- Label cities --
    label_offsets = {
        "Northern Virginia": (12000, 25000),
        "Silicon Valley": (-50000, -40000),
        "Chicago": (15000, 20000),
        "Atlanta": (20000, -25000),
        "Phoenix": (-60000, -30000),
        "Portland/Hillsboro": (15000, 20000),
        "Dallas-Fort Worth": (25000, 20000),
        "Austin": (-70000, -10000),
        "San Antonio": (-85000, -20000),
        "Houston": (20000, -25000),
    }
    for _, row in gdf.iterrows():
        dx, dy = label_offsets.get(row['Market'], (15000, 15000))
        fontsize = 8.5 if 'ERCOT' in row['Tier'] else 7.5
        fontweight = 'bold' if 'ERCOT' in row['Tier'] else 'normal'
        ax.annotate(
            row['Market'], (row.geometry.x, row.geometry.y),
            xytext=(row.geometry.x + dx, row.geometry.y + dy),
            fontsize=fontsize, fontweight=fontweight, color='#2c3e50',
            arrowprops=dict(arrowstyle='-', color='#95a5a6', lw=0.7),
            zorder=10
        )

    # -- Basemap --
    ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)

    # -- ERCOT annotation --
    # Place "ERCOT Service Territory" label inside the dashed box
    ercot_center_x = (gdf[gdf['Tier'].str.contains('ERCOT')].geometry.x.mean())
    ercot_top_y = gdf[gdf['Market'] == 'Dallas-Fort Worth'].geometry.y.values[0] + 120000
    ax.text(ercot_center_x - 100000, ercot_top_y, "ERCOT Service Territory",
            fontsize=10, fontstyle='italic', color='#c0392b', fontweight='bold',
            zorder=10)

    # -- Size legend --
    size_legend_elements = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#95a5a6',
               markersize=np.sqrt(s*0.5), label=f'{s} MW')
        for s in [500, 1500, 4000]
    ]

    # -- Category legend --
    legend1 = ax.legend(
        title="Market Trajectory", loc="lower right",
        fontsize=9.5, title_fontsize=11,
        frameon=True, fancybox=True, framealpha=0.9,
        edgecolor='#bdc3c7'
    )
    for handle in legend1.legend_handles:
        handle._sizes = [120]
    ax.add_artist(legend1)

    # -- Formatting --
    ax.set_axis_off()
    ax.set_title("")  # Title goes in LaTeX caption

    plt.tight_layout()
    plt.savefig("us_texas_datacenters.png", dpi=300, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.close()
    print("Generated: us_texas_datacenters.png")


# ============================================================
# FIGURE 2: Global Hyperscale Data Center Corridors
# ============================================================

def create_global_map():
    global_data = [
        # North America
        {"Region": "N. Virginia / Mid-Atlantic", "Lat": 39.04,  "Lon": -77.49,  "Capacity_MW": 5000, "Cluster": "North America"},
        {"Region": "Texas (ERCOT)",              "Lat": 31.50,  "Lon": -97.00,  "Capacity_MW": 2500, "Cluster": "North America"},
        {"Region": "Silicon Valley / Oregon",    "Lat": 38.50,  "Lon": -121.50, "Capacity_MW": 2500, "Cluster": "North America"},
        {"Region": "Chicago / Midwest",          "Lat": 41.88,  "Lon": -87.63,  "Capacity_MW": 1500, "Cluster": "North America"},
        # Europe (FLAP-D)
        {"Region": "Frankfurt",     "Lat": 50.11,  "Lon": 8.68,   "Capacity_MW": 1400, "Cluster": "Europe (FLAP-D)"},
        {"Region": "London",        "Lat": 51.51,  "Lon": -0.13,  "Capacity_MW": 1600, "Cluster": "Europe (FLAP-D)"},
        {"Region": "Amsterdam",     "Lat": 52.37,  "Lon": 4.90,   "Capacity_MW": 1200, "Cluster": "Europe (FLAP-D)"},
        {"Region": "Paris",         "Lat": 48.86,  "Lon": 2.35,   "Capacity_MW": 900,  "Cluster": "Europe (FLAP-D)"},
        {"Region": "Dublin",        "Lat": 53.35,  "Lon": -6.26,  "Capacity_MW": 1100, "Cluster": "Europe (FLAP-D)"},
        # Asia-Pacific
        {"Region": "Tokyo",         "Lat": 35.68,  "Lon": 139.65, "Capacity_MW": 1500, "Cluster": "Asia-Pacific"},
        {"Region": "Singapore",     "Lat": 1.35,   "Lon": 103.82, "Capacity_MW": 1100, "Cluster": "Asia-Pacific"},
        {"Region": "Sydney",        "Lat": -33.87, "Lon": 151.21, "Capacity_MW": 900,  "Cluster": "Asia-Pacific"},
        {"Region": "Beijing/Hebei", "Lat": 39.90,  "Lon": 116.41, "Capacity_MW": 2200, "Cluster": "Asia-Pacific"},
        # Emerging
        {"Region": "São Paulo",     "Lat": -23.55, "Lon": -46.63, "Capacity_MW": 500,  "Cluster": "Emerging"},
        {"Region": "Mumbai",        "Lat": 19.08,  "Lon": 72.88,  "Capacity_MW": 600,  "Cluster": "Emerging"},
        {"Region": "Johannesburg",  "Lat": -26.20, "Lon": 28.04,  "Capacity_MW": 300,  "Cluster": "Emerging"},
    ]

    df = pd.DataFrame(global_data)
    gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.Lon, df.Lat), crs="EPSG:4326").to_crs(epsg=3857)

    cluster_colors = {
        "North America":    "#2ecc71",
        "Europe (FLAP-D)":  "#3498db",
        "Asia-Pacific":     "#e74c3c",
        "Emerging":         "#f39c12",
    }

    fig, ax = plt.subplots(figsize=(16, 8))

    for cluster, color in cluster_colors.items():
        subset = gdf[gdf['Cluster'] == cluster]
        ax.scatter(
            subset.geometry.x, subset.geometry.y,
            s=subset['Capacity_MW'] * 0.22,
            c=color, alpha=0.75,
            edgecolors='white', linewidth=1,
            label=cluster, zorder=4
        )

    # -- Labels for major hubs --
    label_config = {
        "N. Virginia / Mid-Atlantic": (20000, 40000),
        "Texas (ERCOT)":              (-50000, -50000),
        "Silicon Valley / Oregon":    (-80000, -40000),
        "Chicago / Midwest":          (20000, 30000),
        "Frankfurt":                  (25000, 30000),
        "London":                     (-50000, 35000),
        "Amsterdam":                  (20000, 30000),
        "Dublin":                     (-55000, 30000),
        "Tokyo":                      (25000, 30000),
        "Singapore":                  (25000, -30000),
        "Beijing/Hebei":              (25000, 30000),
        "Sydney":                     (25000, -30000),
        "São Paulo":                  (25000, -35000),
    }

    for _, row in gdf.iterrows():
        if row['Region'] in label_config:
            dx, dy = label_config[row['Region']]
            ax.annotate(
                row['Region'], (row.geometry.x, row.geometry.y),
                xytext=(row.geometry.x + dx, row.geometry.y + dy),
                fontsize=7, color='#2c3e50',
                arrowprops=dict(arrowstyle='-', color='#bdc3c7', lw=0.5),
                zorder=10
            )

    # -- Basemap --
    ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, zoom=2)

    # -- Legend --
    legend = ax.legend(
        title="Data Center Cluster", loc="lower left",
        fontsize=9, title_fontsize=10.5,
        frameon=True, fancybox=True, framealpha=0.9,
        edgecolor='#bdc3c7'
    )
    for handle in legend.legend_handles:
        handle._sizes = [100]

    # -- Formatting --
    ax.set_axis_off()
    ax.set_title("")  # Title goes in LaTeX caption

    plt.tight_layout()
    plt.savefig("global_datacenters.png", dpi=300, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.close()
    print("Generated: global_datacenters.png")


# ============================================================
if __name__ == "__main__":
    create_us_map()
    create_global_map()
    print("\nBoth figures generated successfully.")

In [ ]:
def create_us_texas_map():
    # 1. Prepare US Data
    us_data = [
        {"Market": "Northern Virginia", "State": "VA", "Lat": 39.0438, "Lon": -77.4874, "Capacity_MW": 3500, "Category": "Stable/Saturated", "Color": "gray"},
        {"Market": "Silicon Valley", "State": "CA", "Lat": 37.3541, "Lon": -121.9552, "Capacity_MW": 1500, "Category": "Stable/Saturated", "Color": "gray"},
        {"Market": "Dallas-Fort Worth", "State": "TX", "Lat": 32.7767, "Lon": -96.7970, "Capacity_MW": 1200, "Category": "Hyper-Growth (ERCOT)", "Color": "red"},
        {"Market": "Austin", "State": "TX", "Lat": 30.2672, "Lon": -97.7431, "Capacity_MW": 400, "Category": "Hyper-Growth (ERCOT)", "Color": "red"},
        {"Market": "San Antonio", "State": "TX", "Lat": 29.4241, "Lon": -98.4936, "Capacity_MW": 300, "Category": "Hyper-Growth (ERCOT)", "Color": "red"},
        {"Market": "Houston", "State": "TX", "Lat": 29.7604, "Lon": -95.3698, "Capacity_MW": 250, "Category": "Emerging (ERCOT)", "Color": "darkorange"},
        {"Market": "Chicago", "State": "IL", "Lat": 41.8781, "Lon": -87.6298, "Capacity_MW": 1000, "Category": "Stable/Saturated", "Color": "gray"},
        {"Market": "Atlanta", "State": "GA", "Lat": 33.7490, "Lon": -84.3880, "Capacity_MW": 600, "Category": "High Growth", "Color": "royalblue"},
        {"Market": "Phoenix", "State": "AZ", "Lat": 33.4484, "Lon": -112.0740, "Capacity_MW": 800, "Category": "High Growth", "Color": "royalblue"}
    ]
    df_us = pd.DataFrame(us_data)

    # Convert lat/lon to Geospatial points, then re-project to Web Mercator (EPSG:3857) for the basemap
    gdf_us = gpd.GeoDataFrame(df_us, geometry=gpd.points_from_xy(df_us.Lon, df_us.Lat), crs="EPSG:4326")
    gdf_us = gdf_us.to_crs(epsg=3857)

    # 2. Plot the US Map
    fig, ax = plt.subplots(figsize=(12, 8))

    # Plot each category so we get a clean legend
    for category, data in gdf_us.groupby('Category'):
        color = data['Color'].iloc[0]
        # Size the bubbles based on MW Capacity
        ax.scatter(data.geometry.x, data.geometry.y, s=data['Capacity_MW']*0.6,
                   c=color, alpha=0.75, edgecolors='black', linewidth=1, label=category)

    # Add the professional CartoDB Positron basemap
    ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)

    # Formatting
    ax.set_title("Figure 1: US Data Center Density and ERCOT Hyper-Growth Hubs", fontsize=16, pad=15)
    ax.set_axis_off() # Hides coordinate axes for a clean look

    # Adjust legend
    lgnd = ax.legend(title="Market Trajectory", loc="lower right", fontsize=11, title_fontsize=13)
    # Standardize legend bubble sizes so they don't look weird
    for handle in lgnd.legend_handles:
        handle._sizes = [100]

    # Save to high-res PNG
    plt.tight_layout()
    plt.savefig("us_texas_datacenters_old.png", dpi=300, bbox_inches='tight')
    plt.close()
    print("Successfully generated: us_texas_datacenters.png")






In [ ]:
def create_global_map():
    # 1. Prepare Global Data
    global_data = [
        {"Region": "North America Core", "Lat": 39.8283, "Lon": -98.5795, "Capacity_MW": 10000},
        {"Region": "Frankfurt", "Lat": 50.1109, "Lon": 8.6821, "Capacity_MW": 1300},
        {"Region": "London", "Lat": 51.5074, "Lon": -0.1278, "Capacity_MW": 1500},
        {"Region": "Amsterdam", "Lat": 52.3676, "Lon": 4.9041, "Capacity_MW": 1100},
        {"Region": "Paris", "Lat": 48.8566, "Lon": 2.3522, "Capacity_MW": 800},
        {"Region": "Dublin", "Lat": 53.3498, "Lon": -6.2603, "Capacity_MW": 1200},
        {"Region": "Tokyo", "Lat": 35.6762, "Lon": 139.6503, "Capacity_MW": 1400},
        {"Region": "Singapore", "Lat": 1.3521, "Lon": 103.8198, "Capacity_MW": 1000},
        {"Region": "Sydney", "Lat": -33.8688, "Lon": 151.2093, "Capacity_MW": 850},
        {"Region": "Beijing/Hebei", "Lat": 39.9042, "Lon": 116.4074, "Capacity_MW": 2000}
    ]
    df_global = pd.DataFrame(global_data)

    gdf_global = gpd.GeoDataFrame(df_global, geometry=gpd.points_from_xy(df_global.Lon, df_global.Lat), crs="EPSG:4326")
    gdf_global = gdf_global.to_crs(epsg=3857)

    # 2. Plot the Global Map
    fig, ax = plt.subplots(figsize=(14, 7))

    # Plot bubbles
    ax.scatter(gdf_global.geometry.x, gdf_global.geometry.y, s=gdf_global['Capacity_MW']*0.25,
               c='#2ca02c', alpha=0.7, edgecolors='black', linewidth=1)

    # Add the basemap (Zoom level 2 is perfect for a full world view)
    ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, zoom=2)

    # Formatting
    ax.set_title("Figure 2: Global Hyperscale Data Center Corridors", fontsize=16, pad=15)
    ax.set_axis_off()

    plt.tight_layout()
    plt.savefig("global_datacenters_old.png", dpi=300, bbox_inches='tight')
    plt.close()
    print("Successfully generated: global_datacenters.png")


In [ ]:
if __name__ == "__main__":
    create_us_texas_map()
    create_global_map()